# 01b - Metrik Kesalahan Klasifikasi
Materi Modul 1: matriks kebingungan (confusion matrix), akurasi, spesifisitas, presisi, recall, kurva ROC, dan kurva Presisi-Recall.

Contoh kasus: deteksi kanker. Kelas positif (1) = **ganas**.

| | Prediksi Ganas | Prediksi Jinak |
|---|---|---|
| **Asli Ganas** | TP | FN |
| **Asli Jinak** | FP | TN |

- Akurasi = (TP+TN) / semua
- Presisi = TP / (TP+FP)
- Recall (sensitivitas) = TP / (TP+FN)
- Spesifisitas = TN / (TN+FP)
- F1 = rata-rata harmonik presisi dan recall

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

## 1. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, pred)
print(cm)

ConfusionMatrixDisplay(cm, display_labels=['jinak', 'ganas']).plot(cmap='Blues')
plt.title('Confusion matrix'); plt.show()

## 2. Hitung metrik manual dari confusion matrix

In [ ]:
TN, FP, FN, TP = cm.ravel()
print('TP =', TP, '| TN =', TN, '| FP =', FP, '| FN =', FN)
print()
print('Akurasi     :', round((TP + TN) / (TP + TN + FP + FN), 4))
print('Presisi     :', round(TP / (TP + FP), 4))
print('Recall      :', round(TP / (TP + FN), 4))
print('Spesifisitas:', round(TN / (TN + FP), 4))
f1 = 2 * (TP / (TP + FP)) * (TP / (TP + FN)) / ((TP / (TP + FP)) + (TP / (TP + FN)))
print('F1          :', round(f1, 4))

## 3. Pakai fungsi sklearn (hasilnya harus sama)

In [ ]:
print('Akurasi :', round(accuracy_score(y_test, pred), 4))
print('Presisi :', round(precision_score(y_test, pred), 4))
print('Recall  :', round(recall_score(y_test, pred), 4))
print('F1      :', round(f1_score(y_test, pred), 4))
print()
print(classification_report(y_test, pred, target_names=['jinak', 'ganas']))

## 4. Pengaruh threshold
Default threshold = 0.5. Menurunkan threshold -> recall naik, presisi turun. Untuk kanker, FN (ganas terlewat) lebih berbahaya, jadi recall dikejar.

In [ ]:
print('Threshold | Presisi | Recall | FP | FN')
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    p = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    print(f'{t:<9} | {precision_score(y_test, p):.3f}   | {recall_score(y_test, p):.3f}  | {fp:2d} | {fn:2d}')

## 5. Kurva ROC dan AUC

In [ ]:
fpr, tpr, thr = roc_curve(y_test, proba)
auc = roc_auc_score(y_test, proba)

plt.plot(fpr, tpr, label=f'Regresi Logistik (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', label='Tebakan acak')
plt.xlabel('False Positive Rate (1 - spesifisitas)'); plt.ylabel('True Positive Rate (recall)')
plt.title('Kurva ROC'); plt.legend(); plt.show()

## 6. Kurva Presisi-Recall
Lebih informatif daripada ROC bila kelas positif langka (lihat Modul 6).

In [ ]:
prec, rec, thr_pr = precision_recall_curve(y_test, proba)
ap = average_precision_score(y_test, proba)

plt.plot(rec, prec, label=f'AP={ap:.3f}')
plt.axhline(y_test.mean(), color='red', linestyle='--', label='Baseline (proporsi kelas positif)')
plt.xlabel('Recall'); plt.ylabel('Presisi')
plt.title('Kurva Presisi-Recall'); plt.legend(); plt.show()

## 7. Metrik untuk multi-kelas (Wine)
Ada rata-rata `macro` (tiap kelas sama bobot) dan `weighted` (sesuai jumlah sampel).

In [ ]:
wine = pd.read_csv('../datasets/wine.csv')
Xw, yw = wine.drop(columns='target'), wine['target']
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, stratify=yw, random_state=42)

mw = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xw_tr, yw_tr)
pw = mw.predict(Xw_te)

print(confusion_matrix(yw_te, pw))
print('F1 macro   :', round(f1_score(yw_te, pw, average='macro'), 4))
print('F1 weighted:', round(f1_score(yw_te, pw, average='weighted'), 4))
print('AUC (OvR)  :', round(roc_auc_score(yw_te, mw.predict_proba(Xw_te), multi_class='ovr'), 4))

## Catatan
- (kapan fokus recall vs presisi, kapan pakai ROC vs PR curve, arti AUC, dll.)